<div align="center">
  <a href="https://colab.research.google.com/github/Bengawan-UV-RoboBoat-Team/yolo26-colab/blob/main/YOLO26_train-detect.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>

  # YOLO26: Pipeline Training, Deteksi, dan Ekspor OpenVINO
  ### Bengawan UV RoboBoat Team
  **Target Deployment:** Asus NUC 15 Pro (Intel Core Ultra / Intel Arc GPU & NPU via OpenVINO)
</div>

---
Notebook ini menyediakan alur kerja lengkap untuk siklus pengembangan model deteksi objek maritim RoboBoat:
1. Setup Lingkungan: Instalasi library Ultralytics, OpenVINO, OpenCV, dan verifikasi GPU.
2. Integrasi Google Drive: Sinkronisasi penyimpanan dataset dan penyimpanan langsung checkpoint bobot.
3. Konfigurasi Dataset: Penyiapan file data.yaml untuk misi RoboBoat (avoid_speed, docking, pole, supply).
4. Pelatihan Model: Fine-tuning arsitektur YOLO26 (Nano/Small) dengan logging otomatis.
5. Resume Training: Melanjutkan pelatihan dari checkpoint terakhir (last.pt) jika sesi runtime terputus.
6. Validasi & Evaluasi: Evaluasi metrik mAP50, mAP50-95, precision, recall, dan confusion matrix.
7. Pengujian Deteksi: Inferensi gambar uji inline dengan visualisasi bounding box.
8. Ekspor OpenVINO FP16: Optimasi inferensi untuk akselerasi Intel Arc iGPU pada Asus NUC 15 Pro.
9. Manajemen Artefak: Pengemasan dan pengunduhan bobot model ke komputer lokal.

## 1. Setup Lingkungan dan Dependensi

Instalasi library Ultralytics dan OpenVINO runtime untuk pelatihan dan optimasi inferensi model.

In [ ]:
# Instalasi Ultralytics, OpenVINO runtime, dan NNCF (untuk kuantisasi INT8)
!pip install -q "ultralytics>=8.4.0" "openvino>=2024.0.0" "nncf>=2.14.0" matplotlib opencv-python pyyaml

import ultralytics
import openvino as ov
import torch

print(f"[INFO] Ultralytics version : {ultralytics.__version__}")
print(f"[INFO] OpenVINO version    : {ov.__version__}")
print(f"[INFO] PyTorch version     : {torch.__version__}")
print(f"[INFO] CUDA Tersedia       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] Nama Perangkat GPU  : {torch.cuda.get_device_name(0)}")

# Verifikasi lingkungan Ultralytics
ultralytics.checks()


^C
WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\ACER\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ModuleNotFoundError: No module named 'openvino'


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Menghubungkan Google Drive & Setup Direktori

Menghubungkan Google Drive untuk membaca dataset dan menyimpan checkpoint pelatihan secara otomatis.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Menyiapkan struktur direktori utama di Google Drive
BASE_DRIVE_DIR = "/content/drive/MyDrive/RoboBoat-YOLO26"
DATASETS_DIR = os.path.join(BASE_DRIVE_DIR, "datasets")
CONFIGS_DIR = os.path.join(BASE_DRIVE_DIR, "configs")
RUNS_DIR = os.path.join(BASE_DRIVE_DIR, "runs/detect")

os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(CONFIGS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print(f"[INFO] Direktori kerja Google Drive siap di: {BASE_DRIVE_DIR}")

## 3. Konfigurasi Dataset Misi RoboBoat & Versioning

Menyiapkan dataset dan data.yaml berdasarkan misi dan versi dataset.
- Mendukung file zip Roboflow (otomatis mengenali `train/images`, `valid/images`, dan `data.yaml`).
- Otomatis membuat nama eksperimen unik (`EXPERIMENT_NAME = {MISSION_NAME}_{VERSION_TAG}`) agar bobot model dan grafik evaluasi tiap versi tidak tertimpa.


In [ ]:
#@title Konfigurasi Misi dan Dataset
import yaml
import os
import glob

MISSION_NAME = "pole" #@param ["avoid_speed", "docking", "pole", "supply", "custom", "demo_coco8"]
DATASET_FILENAME = "pole15oct_yolo26.zip" #@param {type:"string"} # Nama file zip di datasets/ (kosongkan atau isi 'auto' untuk deteksi otomatis)
VERSION_TAG = "15oct" #@param {type:"string"} # Tag versi untuk identifikasi runs (misal: 15oct, v2, dataset_baru)
USE_LOCAL_CACHE = True #@param {type:"boolean"}

# Penentuan nama eksperimen unik agar runs & checkpoint tersimpan rapi per versi
if MISSION_NAME == "demo_coco8":
    EXPERIMENT_NAME = "demo_coco8"
else:
    tag = VERSION_TAG.strip() if VERSION_TAG else ""
    EXPERIMENT_NAME = f"{MISSION_NAME}_{tag}" if tag else MISSION_NAME

print(f"[INFO] Misi             : {MISSION_NAME}")
print(f"[INFO] Nama Eksperimen : {EXPERIMENT_NAME}")

# Ekstraksi dataset dari Google Drive ke disk lokal Colab (/content/datasets/{EXPERIMENT_NAME})
if MISSION_NAME != "demo_coco8":
    zip_source = None
    
    # 1. Cek jika nama file spesifik diisi dan ada di Drive
    if DATASET_FILENAME and DATASET_FILENAME.strip().lower() != "auto":
        specified_path = os.path.join(DATASETS_DIR, DATASET_FILENAME.strip())
        if os.path.exists(specified_path):
            zip_source = specified_path
            print(f"[INFO] Menggunakan file dataset yang ditentukan: {DATASET_FILENAME}")
        else:
            print(f"[PERINGATAN] File {specified_path} tidak ditemukan. Mencoba mencari file zip yang cocok...")
            
    # 2. Jika belum ketemu / auto, cari file zip berdasarkan pola nama misi di DATASETS_DIR
    if not zip_source:
        matched_zips = sorted(glob.glob(os.path.join(DATASETS_DIR, f"*{MISSION_NAME}*.zip")), key=os.path.getmtime, reverse=True)
        if matched_zips:
            zip_source = matched_zips[0]
            print(f"[INFO] Menemukan dataset otomatis: {os.path.basename(zip_source)}")
        else:
            default_zip = os.path.join(DATASETS_DIR, f"{MISSION_NAME}.zip")
            zip_source = default_zip

    extract_target = f"/content/datasets/{EXPERIMENT_NAME}" if USE_LOCAL_CACHE else os.path.join(DATASETS_DIR, EXPERIMENT_NAME)
    
    if os.path.exists(zip_source):
        print(f"[INFO] Mengekstrak {zip_source} ke {extract_target}...")
        os.makedirs(extract_target, exist_ok=True)
        !unzip -q -o "{zip_source}" -d "{extract_target}"
        print("[INFO] Ekstraksi selesai.")
    else:
        print(f"[PERINGATAN] File dataset {zip_source} belum ditemukan di Google Drive.")
        print(f"[PETUNJUK] Taruh file zip dataset di folder Drive: {DATASETS_DIR}/")

# Definisi mapping kelas standar RoboBoat (fallback)
ROBOBOAT_CLASSES = {
    "avoid_speed": ["blackball", "blackboat", "greenball", "greenlight", "orangeboat", "redball", "redlight", "trapesium", "yellowball"],
    "docking": ["blackboat", "greenlight", "num1", "num2", "num3", "orangeboat", "redlight"],
    "pole": ["green_pole", "red_pole"],
    "supply": ["blackboat", "orangeboat"]
}

# Membuat atau menyelaraskan konfigurasi data.yaml
if MISSION_NAME == "demo_coco8":
    DATASET_YAML_PATH = "coco8.yaml"
else:
    yaml_target = os.path.join(CONFIGS_DIR, f"{EXPERIMENT_NAME}.yaml")
    roboflow_yaml = os.path.join(extract_target, "data.yaml")
    
    if os.path.exists(roboflow_yaml):
        # Gunakan data.yaml bawaan Roboflow yang sudah akurat
        with open(roboflow_yaml, "r") as f:
            data_cfg = yaml.safe_load(f) or {}
            
        data_cfg["path"] = extract_target
        for split in ["train", "val", "test"]:
            if split in data_cfg and isinstance(data_cfg[split], str):
                data_cfg[split] = data_cfg[split].replace("../", "")
        
        with open(yaml_target, "w") as f:
            yaml.dump(data_cfg, f, sort_keys=False)
            
        DATASET_YAML_PATH = yaml_target
        print(f"[INFO] Konfigurasi data.yaml (Roboflow) siap di: {DATASET_YAML_PATH}")
        print(f"       Kelas terdeteksi: {data_cfg.get('names')}")
    else:
        # Fallback pembuatan data.yaml jika tidak ada di dalam zip
        classes = ROBOBOAT_CLASSES.get(MISSION_NAME, ["object"])
        
        val_folder = "valid/images" if os.path.exists(os.path.join(extract_target, "valid")) else "val/images"
        train_folder = "train/images" if os.path.exists(os.path.join(extract_target, "train")) else "images/train"
        test_folder = "test/images" if os.path.exists(os.path.join(extract_target, "test")) else ""
            
        config_dict = {
            "path": extract_target,
            "train": train_folder,
            "val": val_folder,
            "test": test_folder,
            "names": {i: name for i, name in enumerate(classes)}
        }
        with open(yaml_target, "w") as f:
            yaml.dump(config_dict, f, sort_keys=False)
            
        DATASET_YAML_PATH = yaml_target
        print(f"[INFO] File data.yaml berhasil dibuat di: {DATASET_YAML_PATH}")


## 4. Logger Pelatihan (Opsional)

Konfigurasi TensorBoard atau Weights & Biases untuk memantau metrik pelatihan.

In [ ]:
#@title Pengaturan Logger {run: 'auto'}
logger = 'TensorBoard' #@param ['TensorBoard', 'Weights & Biases', 'None']

if logger == 'TensorBoard':
    !yolo settings tensorboard=True
    %load_ext tensorboard
    %tensorboard --logdir {RUNS_DIR}
elif logger == 'Weights & Biases':
    !yolo settings wandb=True
    print("[INFO] Weights & Biases diaktifkan.")
else:
    !yolo settings tensorboard=False wandb=False
    print("[INFO] Logger dinonaktifkan.")

## 5. Train Model YOLO26

Pelatihan model menggunakan bobot awal YOLO26. Checkpoint bobot disimpan langsung ke direktori Google Drive:
- last.pt: Checkpoint epoch terakhir (diperbarui setiap 1 epoch).
- best.pt: Checkpoint bobot dengan evaluasi mAP tertinggi.

In [ ]:
#@title Parameter Pelatihan YOLO26
MODEL_VARIANT = "yolo26n.pt" #@param ["yolo26n.pt", "yolo26s.pt", "yolo26m.pt"]
EPOCHS = 300 #@param {type:"integer"}
PATIENCE = 50 #@param {type:"integer"} # Early stopping: berhenti jika dalam N epoch tidak ada peningkatan mAP (0 = nonaktif)
IMGSZ = "384,640" #@param {type:"string"} # Format: '384,640' [Height, Width] atau '640'
BATCH = 16 #@param {type:"integer"}
SAVE_PERIOD = 10 #@param {type:"integer"}

# Helper tuple imgsz untuk inferensi Python
IMGSZ_TUPLE = tuple(map(int, IMGSZ.split(","))) if "," in str(IMGSZ) else int(IMGSZ)

# Eksekusi pelatihan CLI Ultralytics
!yolo train \
    model={MODEL_VARIANT} \
    data={DATASET_YAML_PATH} \
    epochs={EPOCHS} \
    patience={PATIENCE} \
    imgsz={IMGSZ} \
    batch={BATCH} \
    save_period={SAVE_PERIOD} \
    project={RUNS_DIR} \
    name={EXPERIMENT_NAME} \
    exist_ok=True


## 6. Melanjutkan Pelatihan (Resume Training)

Melanjutkan pelatihan dari checkpoint last.pt yang tersimpan di Google Drive jika proses terputus.

In [ ]:
# Menentukan path checkpoint terakhir di Google Drive
last_weights_path = f"{RUNS_DIR}/{EXPERIMENT_NAME}/weights/last.pt"

if os.path.exists(last_weights_path):
    print(f"[INFO] Melanjutkan pelatihan dari checkpoint: {last_weights_path}")
    !yolo train resume model={last_weights_path}
else:
    print(f"[PERINGATAN] File checkpoint {last_weights_path} tidak ditemukan.")
    print("[PETUNJUK] Pastikan proses training sebelumnya telah berjalan minimal 1 epoch.")


## 7. Validasi dan Evaluasi Model

Evaluasi performa bobot best.pt pada dataset validasi serta menampilkan kurva metrik dan matriks konfusi.

In [ ]:
from IPython.display import Image, display
import os

best_weights_path = f"{RUNS_DIR}/{EXPERIMENT_NAME}/weights/best.pt"

# 1. Jalankan validasi pada bobot terbaik hasil pelatihan
!yolo val \
    model={best_weights_path} \
    data={DATASET_YAML_PATH} \
    imgsz={IMGSZ}

# 2. Tampilkan grafik hasil pelatihan
training_plots = [
    f"{RUNS_DIR}/{EXPERIMENT_NAME}/results.png",
    f"{RUNS_DIR}/{EXPERIMENT_NAME}/confusion_matrix.png",
    f"{RUNS_DIR}/{EXPERIMENT_NAME}/val_batch0_pred.jpg"
]

print("\n--- Grafik Evaluasi Pelatihan ---")
for plot_path in training_plots:
    if os.path.exists(plot_path):
        print(f"[INFO] Menampilkan: {plot_path}")
        display(Image(plot_path, width=800))


## 8. Pengujian Deteksi Visual

Uji inferensi model menggunakan OpenCV dan Matplotlib pada gambar uji.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Memuat model terbaik dari Google Drive
model = YOLO(best_weights_path)

TEST_IMAGE_SOURCE = "https://ultralytics.com/images/bus.jpg" #@param {type:"string"}
CONF_THRESHOLD = 0.25 #@param {type:"slider", min:0.1, max:1.0, step:0.05}

# Menjalankan prediksi dengan ukuran imgsz yang sesuai
results = model.predict(source=TEST_IMAGE_SOURCE, conf=CONF_THRESHOLD, imgsz=IMGSZ_TUPLE)

# Menampilkan hasil visual inline
for i, r in enumerate(results):
    res_bgr = r.plot()
    res_rgb = cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(res_rgb)
    plt.title(f"YOLO26 Detection - {EXPERIMENT_NAME} (conf={CONF_THRESHOLD})")
    plt.axis("off")
    plt.show()
    
    # Ringkasan deteksi
    boxes = r.boxes
    print(f"[HASIL] Jumlah objek terdeteksi: {len(boxes)}")
    for box in boxes:
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]
        score = float(box.conf[0])
        print(f" - {cls_name}: {score:.2%}")


## 9. Ekspor ke OpenVINO (Deployment Asus NUC 15 Pro)

Konversi bobot best.pt ke format OpenVINO untuk akselerasi inferensi pada prosesor Intel Core Ultra, Intel Arc iGPU, dan NPU.

| Format / Presisi | Opsi CLI | Target Hardware NUC 15 Pro | Karakteristik |
|---|---|---|---|
| **OpenVINO (INT8)** | `int8=True data=data.yaml` | **Intel NPU / CPU / Arc iGPU** | Model teringan (~2-3 MB), latensi terendah, sangat hemat daya untuk onboard robot |
| **OpenVINO (FP16)** | `half=True` | **Intel Arc iGPU** | Rekomendasi grafis Intel Arc, akurasi penuh tanpa kalibrasi dataset |
| **OpenVINO (FP32)** | `half=False` | **Intel CPU standar** | Format dasar 32-bit float |
| **ONNX** | `format=onnx` | Cross-Platform | Format portabel universal |


In [ ]:
#@title Ekspor Bobot ke OpenVINO
PRECISION = "INT8" #@param ["INT8", "FP16", "FP32"]
EXPORT_IMGSZ = "384,640" #@param {type:"string"}

extra_args = []
if PRECISION == "INT8":
    # INT8 membutuhkan calibration dataset dari data.yaml via NNCF
    extra_args.append(f"int8=True data={DATASET_YAML_PATH}")
elif PRECISION == "FP16":
    extra_args.append("half=True")
else:
    extra_args.append("half=False")

extra_args_str = " ".join(extra_args)

# Ekspor bobot terbaik ke OpenVINO
!yolo export \
    model={best_weights_path} \
    format=openvino \
    {extra_args_str} \
    imgsz={EXPORT_IMGSZ}

# Verifikasi direktori OpenVINO
import os
import glob

weights_dir = f"{RUNS_DIR}/{EXPERIMENT_NAME}/weights"
ov_dirs = glob.glob(f"{weights_dir}/*openvino*")
print(f"[INFO] Direktori OpenVINO ditemukan: {ov_dirs}")
for ov_dir in ov_dirs:
    print(f"[INFO] Berkas di dalam {ov_dir}:", os.listdir(ov_dir))


In [ ]:
# Verifikasi inferensi model OpenVINO
import os
import glob
from ultralytics import YOLO

weights_dir = f"{RUNS_DIR}/{EXPERIMENT_NAME}/weights"
ov_dirs = sorted(glob.glob(f"{weights_dir}/*openvino*"), key=os.path.getmtime, reverse=True)
openvino_model_path = ov_dirs[0] if ov_dirs else f"{weights_dir}/best_openvino_model/"

EXPORT_IMGSZ_TUPLE = tuple(map(int, EXPORT_IMGSZ.split(","))) if "," in str(EXPORT_IMGSZ) else int(EXPORT_IMGSZ)

if os.path.exists(openvino_model_path):
    print(f"[INFO] Memuat model OpenVINO dari {openvino_model_path}...")
    ov_model = YOLO(openvino_model_path)
    ov_results = ov_model.predict(source=TEST_IMAGE_SOURCE, conf=CONF_THRESHOLD, imgsz=EXPORT_IMGSZ_TUPLE)
    print("[INFO] Inferensi runtime OpenVINO berhasil dijalankan.")
else:
    print(f"[PERINGATAN] Direktori {openvino_model_path} belum ditemukan. Jalankan cell ekspor terlebih dahulu.")


## 10. Pengemasan dan Pengunduhan Artefak (Opsional)

Kompresi dan pengunduhan arsip zip model OpenVINO ke komputer lokal untuk ditransfer ke Asus NUC 15 Pro.

In [ ]:
import os
import shutil
import glob
from google.colab import files

weights_dir = f"{RUNS_DIR}/{EXPERIMENT_NAME}/weights"
ov_dirs = sorted(glob.glob(f"{weights_dir}/*openvino*"), key=os.path.getmtime, reverse=True)
ov_model_dir = ov_dirs[0] if ov_dirs else f"{weights_dir}/best_openvino_model"
folder_name = os.path.basename(os.path.normpath(ov_model_dir))
zip_name = f"{EXPERIMENT_NAME}_{folder_name}"

# Mengompres dan mengunduh model OpenVINO
if os.path.exists(ov_model_dir):
    print(f"[INFO] Mengompres {ov_model_dir}...")
    shutil.make_archive(zip_name, "zip", ov_model_dir)
    print(f"[INFO] Mengunduh {zip_name}.zip...")
    files.download(f"{zip_name}.zip")
else:
    print(f"[PERINGATAN] Direktori {ov_model_dir} tidak ditemukan.")
